# 01. Actualizaci?n monetaria mediante INPC

Se actualizan importes anuales a diciembre de 2025. Los pagos por desarrollo no se interpretan como acumulados sin confirmaci?n.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "funciones_reservas.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontr? la ra?z del proyecto.")
sys.path.insert(0, str(ROOT / "src"))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
from funciones_reservas import (ajustar_inpc, cargar_excel, guardar_snapshot,
                                normalizar_categorias)
archivo = RAW / "2026 BASE RESERVAS.xlsx"
emision = cargar_excel(archivo, "EMISION")
siniestros = cargar_excel(archivo, "SINIESTRALIDAD")
for df in [emision, siniestros]:
    columnas = [col for col in ["tipo_embarcacion", "uso"] if col in df]
    if columnas:
        df.loc[:, columnas] = normalizar_categorias(df, columnas)[columnas]


## Serie INPC y criterio temporal

In [ ]:
serie_inpc = None
try:
    import banxicolab as sie
    serie_inpc = pd.Series(sie.descargar("inpc", inicio="2015-01-01", fin="2025-12-31")["inpc"])
    print("INPC descargado expl?citamente para 2015-2025.")
except ImportError:
    print("Falta banxicolab. Inst?lalo o proporciona una serie mensual de INPC antes de ajustar.")
except Exception as exc:
    print(f"No se pudo obtener el INPC: {exc}. Revisa el token o proporciona una serie local.")



### 1.1 Ajustar los valores Mediante el INPC

Pasos:
1. Configurar el token de Banxico 
2. Descargar la serie mensual del INPC
3. Ajustar los valores mediante el INPC


La fecha objetivo es diciembre de 2025; para registros anuales se usa diciembre del a?o de origen.

In [ ]:
if serie_inpc is not None:
    emision_ajustada = ajustar_inpc(
        emision,
        ["prima_emitida", "prima_retenida", "comision_directa"],
        "a?o", 2025, serie_inpc, frecuencia="anual", criterio_anual="diciembre"
    )
    siniestros_ajustados = siniestros.copy()
    for desarrollo in range(4):
        columna = f"{desarrollo}_monto pagado"
        siniestros_ajustados["a?o_pago"] = siniestros_ajustados["a?o"] + desarrollo
        mascara = siniestros_ajustados["a?o_pago"] <= 2025
        datos = ajustar_inpc(
            siniestros_ajustados.loc[mascara, ["a?o_pago", columna]],
            columna, "a?o_pago", 2025, serie_inpc,
            frecuencia="anual", criterio_anual="diciembre"
        )
        siniestros_ajustados[f"{columna}_ajustada"] = datos[f"{columna}_ajustada"]
    siniestros_ajustados.drop(columns="a?o_pago", inplace=True)
    print("Criterio: a?o de origen para emisi?n; a?o de pago = origen + desarrollo para siniestros.")
    print("Los desarrollos se conservan como columnas observadas; no se supone que sean acumulados.")
    display(emision_ajustada.head(), siniestros_ajustados.head())
    guardar_snapshot(
        emision_ajustada, PROCESSED / "emision_ajustada_2025.parquet",
        periodo_valuacion="2025-12", fuente_inpc="Banxico/serie mensual",
        criterio_actualizacion="Diciembre de cada a?o a diciembre de 2025",
        archivos_origen=[archivo]
    )
    guardar_snapshot(
        siniestros_ajustados, PROCESSED / "siniestros_ajustados_2025.parquet",
        periodo_valuacion="2025-12", fuente_inpc="Banxico/serie mensual",
        criterio_actualizacion="Diciembre del a?o de pago a diciembre de 2025",
        archivos_origen=[archivo]
    )
else:
    print("No se generaron ajustes ni snapshots porque falta una serie INPC v?lida.")
